# Chapitre 4. Espérance, variance, moments, inégalités

Prime pure, variance, prime avec franchise et espérance conditionnelle : les briques de la TVaR.

*Cours « Probabilités pour la gestion des risques », M1 GRAF. Exécutez les cellules dans l'ordre (Maj + Entrée).*

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, integrate
from math import comb, perm, factorial, log, exp

rng = np.random.default_rng(42)   # graine : memes resultats a chaque execution

## Espérance et variance d'une loi discrète (fil rouge)

In [2]:
k = np.array([0, 1, 2, 3]); p = np.array([0.60, 0.25, 0.10, 0.05])
E = np.sum(k * p); E2 = np.sum(k**2 * p)
print('E[N] =', E, '| E[N^2] =', E2, '| Var =', E2 - E**2)

F = np.cumsum(p)
print('E[N] par la survie : somme des P(N > k) =', np.sum(1 - F[:-1]))

E[N] = 0.6000000000000001 | E[N^2] = 1.1 | Var = 0.74
E[N] par la survie : somme des P(N > k) = 0.6000000000000001


## Exemple résolu : Contrat avec franchise (exponentielle de moyenne 500)

In [3]:
X = stats.expon(scale=500)
print('E, Var, sigma :', X.mean(), X.var(), X.std(), '| asymetrie :', X.stats(moments='s'))

stop_loss, _ = integrate.quad(X.sf, 200, np.inf)
print('Prime pure avec franchise 200 =', stop_loss, '| formule 500 e^-0.4 =', 500 * np.exp(-0.4))

num, _ = integrate.quad(lambda t: t * X.pdf(t), 1000, np.inf)
print('E[X | X > 1000] =', num / X.sf(1000))

sim = X.rvs(size=1_000_000, random_state=rng)
print('Simulation : franchise ~', np.maximum(sim - 200, 0).mean(), '| E[X|X>1000] ~', sim[sim > 1000].mean())

E, Var, sigma : 500.0 250000.0 500.0 | asymetrie : 2.0
Prime pure avec franchise 200 = 335.1600230178197 | formule 500 e^-0.4 = 335.16002301781964
E[X | X > 1000] = 1500.0
Simulation : franchise ~ 335.0316177483248 | E[X|X>1000] ~ 1499.112444412913


## Markov et Tchebychev face à la vraie valeur

In [4]:
for a in [1000, 2000, 5000]:
    print(f'a = {a} : Markov <= {500/a:.3f} | vraie valeur (exponentielle) = {X.sf(a):.6f}')

a = 1000 : Markov <= 0.500 | vraie valeur (exponentielle) = 0.135335
a = 2000 : Markov <= 0.250 | vraie valeur (exponentielle) = 0.018316
a = 5000 : Markov <= 0.100 | vraie valeur (exponentielle) = 0.000045


## Fonction génératrice des moments (SymPy)

In [5]:
import sympy as sp
t, lam = sp.symbols('t lambda', positive=True)
x = sp.symbols('x', positive=True)
M = sp.integrate(sp.exp(t*x) * lam * sp.exp(-lam*x), (x, 0, sp.oo), conds='none')
M = sp.simplify(M)
print('M(t) =', M)
print('E[X] = M\'(0) =', sp.simplify(sp.diff(M, t).subs(t, 0)))
print('E[X^2] = M\'\'(0) =', sp.simplify(sp.diff(M, t, 2).subs(t, 0)))

M(t) = lambda/(lambda - t)
E[X] = M'(0) = 1/lambda
E[X^2] = M''(0) = 2/lambda**2


## Vérification des exercices 4.1 à 4.6

In [6]:
p41 = np.array([0.5, 0.3, 0.15, 0.05]); e = np.sum(k * p41)
print('4.1 :', e, '| par la survie', np.sum(1 - np.cumsum(p41)[:-1]), '| Var =', np.sum(k**2 * p41) - e**2)

f = lambda u: 2 * u
E42, _ = integrate.quad(lambda u: u * f(u), 0, 1); E42b, _ = integrate.quad(lambda u: u**2 * f(u), 0, 1)
print('4.2 :', E42, E42b, E42b - E42**2)

U = stats.uniform(loc=3, scale=5)   # uniforme sur [3, 8]
print('4.3 : uniforme [3,8] -> E =', U.mean(), '=(3+8)/2 | Var =', U.var(), '= 25/12 =', 25/12)

print('4.4 : Markov', 1000/2000, '| Tchebychev', (300/1000)**2)

p_b = sp.symbols('p', positive=True)
Mb = 1 - p_b + p_b * sp.exp(t)
print('4.5 : E =', sp.diff(Mb, t).subs(t, 0), '| Var =', sp.factor(sp.diff(Mb, t, 2).subs(t, 0) - sp.diff(Mb, t).subs(t, 0)**2))

Y = stats.expon(scale=2000)
print('4.6 exact : (a)', Y.mean(), Y.var(), '(b)', 2000 * np.exp(-0.25), '(c)', 2000 * (1 - np.exp(-2.5)), '(d)', 6000)
s = Y.rvs(size=1_000_000, random_state=rng)
print('4.6 simul : (b)', np.maximum(s - 500, 0).mean(), '(c)', np.minimum(s, 5000).mean(), '(d)', s[s > 4000].mean())

4.1 : 0.75 | par la survie 0.7499999999999999 | Var = 0.7874999999999999
4.2 : 0.6666666666666667 0.5 0.05555555555555547
4.3 : uniforme [3,8] -> E = 5.5 =(3+8)/2 | Var = 2.083333333333333 = 25/12 = 2.0833333333333335
4.4 : Markov 0.5 | Tchebychev 0.09
4.5 : E = p | Var = -p*(p - 1)
4.6 exact : (a) 2000.0 4000000.0 (b) 1557.6015661428098 (c) 1835.8300027522023 (d) 6000
4.6 simul : (b) 1555.304430715262 (c) 1834.1800730814987 (d) 6001.04825276668
